In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

In [ ]:
sentiment = pd.read_csv("../data/fear_greed_index.csv")
trades = pd.read_csv("../data/historical_data.csv")

print("Sentiment shape:", sentiment.shape)
print("Trades shape:", trades.shape)

In [ ]:
sentiment.columns, trades.columns

In [ ]:
sentiment.head()

In [ ]:
trades.head()

In [ ]:
# Convert date
sentiment['date'] = pd.to_datetime(sentiment['date'])
sentiment['Date'] = sentiment['date'].dt.date

# Simplify sentiment
def simplify(x):
    return 'Fear' if 'Fear' in x else 'Greed'

sentiment['sentiment'] = sentiment['classification'].apply(simplify)

# Keep only required columns
sentiment = sentiment[['Date', 'sentiment']]

sentiment.head()

In [ ]:
trades['Timestamp IST'] = pd.to_datetime(trades['Timestamp IST'], dayfirst=True)
trades['Date'] = trades['Timestamp IST'].dt.date

In [ ]:
trades.head()


In [ ]:
# Rename columns
trades = trades.rename(columns={
    'Account': 'account',
    'Execution Price': 'execution_price',
    'Size Tokens': 'size_tokens',
    'Size USD': 'size_usd',
    'Side': 'side',
    'Closed PnL': 'closedPnL',
    'Fee': 'fee'
})

In [ ]:
df = trades.merge(sentiment, on='Date', how='left')
df['sentiment'].isnull().sum()

In [ ]:
df['win'] = df['closedPnL'] > 0


In [ ]:
df.groupby('sentiment')['closedPnL'].describe()

In [ ]:
df.groupby('sentiment')['win'].mean()


In [ ]:
sns.boxplot(x='sentiment', y='closedPnL', data=df)
plt.title("PnL vs Sentiment")
plt.show()

In [ ]:
sns.barplot(x='sentiment', y='win', data=df)
plt.title("Win Rate vs Sentiment")
plt.show()

In [ ]:
df.groupby('sentiment')['size_usd'].mean()

In [ ]:
df.groupby('sentiment').size()

In [ ]:
df.groupby('sentiment')['closedPnL'].std()

In [ ]:
median_size = df['size_usd'].median()

df['size_group'] = df['size_usd'].apply(
    lambda x: 'High' if x > median_size else 'Low'
)

df.groupby(['size_group', 'sentiment'])['closedPnL'].mean()

In [ ]:
# PnL distribution
pnl_stats = df.groupby('sentiment')['closedPnL'].describe()

# Win rate
win_rate = df.groupby('sentiment')['win'].mean()

# Drawdown proxy (use lower tail / min)
drawdown = df.groupby('sentiment')['closedPnL'].min()

pnl_stats, win_rate, drawdown

In [ ]:
# Visuals
import matplotlib.pyplot as plt
import seaborn as sns

sns.boxplot(x='sentiment', y='closedPnL', data=df)
plt.title("PnL Distribution by Sentiment")
plt.show()

sns.barplot(x=win_rate.index, y=win_rate.values)
plt.title("Win Rate by Sentiment")
plt.ylabel("Win Rate")
plt.show()

In [ ]:
# Position size (risk proxy)
size_mean = df.groupby('sentiment')['size_usd'].mean()

# Trade frequency
trade_count = df.groupby('sentiment').size()

# Volatility
volatility = df.groupby('sentiment')['closedPnL'].std()

size_mean, trade_count, volatility

In [ ]:
# Charts
sns.barplot(x=size_mean.index, y=size_mean.values)
plt.title("Avg Position Size by Sentiment")
plt.ylabel("USD")
plt.show()

sns.barplot(x=trade_count.index, y=trade_count.values)
plt.title("Trade Count by Sentiment")
plt.show()

In [ ]:
counts = df['account'].value_counts()
freq_accounts = counts[counts > counts.median()].index

df['freq_group'] = df['account'].apply(lambda x: 'Frequent' if x in freq_accounts else 'Infrequent')

df.groupby(['freq_group', 'sentiment'])['closedPnL'].mean()

In [ ]:
seg1 = df.groupby(['freq_group', 'sentiment'])['closedPnL'].mean().reset_index()

sns.barplot(x='freq_group', y='closedPnL', hue='sentiment', data=seg1)
plt.title("PnL: Frequent vs Infrequent Traders")
plt.show()

In [ ]:
total_pnl = df.groupby('account')['closedPnL'].sum()
winners = total_pnl[total_pnl > 0].index

df['trader_type'] = df['account'].apply(lambda x: 'Winner' if x in winners else 'Loser')

df.groupby(['trader_type', 'sentiment'])['closedPnL'].mean()

In [ ]:
seg2 = df.groupby(['trader_type', 'sentiment'])['closedPnL'].mean().reset_index()

sns.barplot(x='trader_type', y='closedPnL', hue='sentiment', data=seg2)
plt.title("PnL: Winners vs Losers")
plt.show()

In [ ]:
median_size = df['size_usd'].median()

df['size_group'] = df['size_usd'].apply(lambda x: 'High' if x > median_size else 'Low')

df.groupby(['size_group', 'sentiment'])['closedPnL'].mean()

In [ ]:
seg3 = df.groupby(['size_group', 'sentiment'])['closedPnL'].mean().reset_index()

sns.barplot(x='size_group', y='closedPnL', hue='sentiment', data=seg3)
plt.title("PnL: High vs Low Position Size")
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# Create label: profitable trade
df['profit_label'] = (df['closedPnL'] > 0).astype(int)

# Encode sentiment
df['sentiment_enc'] = df['sentiment'].map({'Fear': 0, 'Greed': 1})

features = ['size_usd', 'sentiment_enc']
X = df[features]
y = df['profit_label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = RandomForestClassifier()
model.fit(X_train, y_train)

print("Accuracy:", model.score(X_test, y_test))

In [ ]:
# Daily aggregates per trader
daily = df.groupby(['account', 'Date']).agg({
    'closedPnL': 'sum',
    'size_usd': 'mean',
    'win': 'mean'
}).reset_index()

sent_map = df[['Date', 'sentiment']].drop_duplicates()
daily = daily.merge(sent_map, on='Date', how='left')

# Encode sentiment
daily['sentiment_enc'] = daily['sentiment'].map({'Fear': 0, 'Greed': 1})

daily.head()

In [ ]:
# Next-day PnL
daily['next_pnl'] = daily.groupby('account')['closedPnL'].shift(-1)

# Bucket: 1 = profitable next day, 0 = not
daily['next_profit_label'] = (daily['next_pnl'] > 0).astype(int)

In [ ]:
# Absolute change as a simple volatility proxy
daily['pnl_volatility'] = daily.groupby('account')['closedPnL'].diff().abs()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

features = ['closedPnL', 'size_usd', 'win', 'sentiment_enc']
data = daily.dropna(subset=features + ['next_profit_label'])

X = data[features]
y = data['next_profit_label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = RandomForestClassifier()
model.fit(X_train, y_train)

print("Accuracy:", model.score(X_test, y_test))

In [ ]:
from sklearn.ensemble import RandomForestRegressor

data_reg = daily.dropna(subset=features + ['pnl_volatility'])

X = data_reg[features]
y = data_reg['pnl_volatility']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

reg = RandomForestRegressor()
reg.fit(X_train, y_train)

print("R2 Score:", reg.score(X_test, y_test))

In [ ]:
df.to_csv("../data/merged.csv", index=False)